# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [9]:
print("Unit of analysis: one row = one content item's performance on one report_date, for one client")
print("Time window used for this contract: month=2026-03 (2026-03-01 to 2026-03-31), a mid-panel month")
print("Row count verified: 9,841,378 rows")

Unit of analysis: one row = one content item's performance on one report_date, for one client
Time window used for this contract: month=2026-03 (2026-03-01 to 2026-03-31), a mid-panel month
Row count verified: 9,841,378 rows


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [10]:
!pip install duckdb -q

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [11]:
print("FEATURES (knowable at prediction time, safe to use):")
print("- gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position")
print("- ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec")
print("- sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai")
print("- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other")
print("- scroll_events")
print("- client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available (availability flags)")
print()
print("LABEL / PROXY:")
print("- None exists directly in this warehouse table. Unlike the starter CSV (which has")
print("  is_declining_label), this fact table is raw daily performance only.")
print("- If a label is needed later, it would have to be built as a proxy by comparing the")
print("  same client_hash_id + content_hash_id across two DIFFERENT months (e.g. March vs April),")
print("  never within the same row. This avoids the same trap the starter CSV warns about.")
print()
print("CONTEXT (grouping/joining/reading only, never features):")
print("- client_hash_id, content_hash_id — pseudonyms, used to group/join only")
print("- report_date — defines the time window, not a feature itself")
print("- month — partition key, redundant with report_date")
print()
print("EXCLUDED:")
print("- None of these 31 columns are private or future-only within this single month's slice.")
print("- If building a label from a LATER month, that later month's rows would be excluded from")
print("  the feature set to prevent leakage across the prediction boundary.")

FEATURES (knowable at prediction time, safe to use):
- gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position
- ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec
- sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai
- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other
- scroll_events
- client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available (availability flags)

LABEL / PROXY:
- None exists directly in this warehouse table. Unlike the starter CSV (which has
  is_declining_label), this fact table is raw daily performance only.
- If a label is needed later, it would have to be built as a proxy by comparing the
  same client_hash_id + content_hash_id across two DIFFERENT months (e.g. March vs April),
  never within the same row. This avoids the same trap the starter CSV warns about.

CONTEXT (grouping/joining/reading only, never features):
- client_hash_id

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
# Grain check: one row per client + content + date?
con.sql("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()

# Counts + window
con.sql("""
    SELECT COUNT(*) AS row_count,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").show()

# Missingness: GA4 availability
con.sql("""
    SELECT
        AVG(CASE WHEN ga4_data_available THEN 1.0 ELSE 0 END) AS pct_ga4_available,
        AVG(CASE WHEN gsc_data_available THEN 1.0 ELSE 0 END) AS pct_gsc_available
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────┐
│ client_hash_id │ content_hash_id │ report_date │   c   │
│    varchar     │     varchar     │    date     │ int64 │
├────────────────┴─────────────────┴─────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬───────────┬───────────┬────────────┬────────────┐
│ row_count │ n_clients │ n_content │  min_date  │  max_date  │
│   int64   │   int64   │   int64   │    date    │    date    │
├───────────┼───────────┼───────────┼────────────┼────────────┤
│   9841378 │        55 │    331437 │ 2026-03-01 │ 2026-03-31 │
└───────────┴───────────┴───────────┴────────────┴────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────┬─────────────────────┐
│  pct_ga4_available  │  pct_gsc_available  │
│       double        │       double        │
├─────────────────────┼─────────────────────┤
│ 0.04206382480177065 │ 0.36692635929643186 │
└─────────────────────┴─────────────────────┘



In [13]:
print("FIVE-FEATURE FRAME — Content performance signals for March 2026")
print()
print("1. gsc_avg_position — knowable at decision moment because: it's a completed daily")
print("   Search Console measurement, available same-day.")
print()
print("2. ga4_engaged_sessions — knowable at decision moment because: GA4 logs sessions as they")
print("   happen; no future data needed, only filtered by ga4_data_available=True.")
print()
print("3. sessions_ai — knowable at decision moment because: AI-referral sessions are counted")
print("   same-day, same as other traffic sources.")
print()
print("4. scroll_events — knowable at decision moment because: an observed on-page engagement")
print("   signal logged as it occurs, no lag.")
print()
print("5. gsc_clicks — knowable at decision moment because: a same-day completed Search")
print("   Console metric, not a future outcome.")

FIVE-FEATURE FRAME — Content performance signals for March 2026

1. gsc_avg_position — knowable at decision moment because: it's a completed daily
   Search Console measurement, available same-day.

2. ga4_engaged_sessions — knowable at decision moment because: GA4 logs sessions as they
   happen; no future data needed, only filtered by ga4_data_available=True.

3. sessions_ai — knowable at decision moment because: AI-referral sessions are counted
   same-day, same as other traffic sources.

4. scroll_events — knowable at decision moment because: an observed on-page engagement
   signal logged as it occurs, no lag.

5. gsc_clicks — knowable at decision moment because: a same-day completed Search
   Console metric, not a future outcome.


In [14]:
leaky = con.sql("""
    SELECT client_hash_id, content_hash_id, gsc_clicks
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available = TRUE
    LIMIT 20000
""").df()

# Median split gives a roughly balanced 50/50 label
median_clicks = leaky['gsc_clicks'].median()
leaky['is_high_performer'] = (leaky['gsc_clicks'] > median_clicks).astype(int)
leaky['leaky_feature'] = leaky['gsc_clicks']

print("Class balance check:")
print(leaky['is_high_performer'].value_counts())
print()


correlation = leaky['leaky_feature'].corr(leaky['is_high_performer'])
print(f"Correlation between leaky_feature and the label: {correlation:.3f}")
print("A correlation this high (0.6) between a single raw feature and a label is a red flag —")
print("real, independent signals rarely correlate this strongly with an outcome. It's high")
print("because gsc_clicks was literally used to construct the label via a threshold, not")
print("because it's a genuinely powerful predictor.")
print("This is artificially inflated because 'leaky_feature' (gsc_clicks) is the exact")
print("value the label was thresholded from — a real model would 'cheat' by learning this")
print("shortcut instead of a genuine pattern.")
print()
print("STEP 2: Remove the leaky feature and keep only honest, independent features:")
honest_features = ['gsc_avg_position', 'ga4_engaged_sessions', 'sessions_ai', 'scroll_events']
print(f"Honest feature set: {honest_features}")

Class balance check:
is_high_performer
0    17497
1     2503
Name: count, dtype: int64

Correlation between leaky_feature and the label: 0.599
A correlation this high (0.6) between a single raw feature and a label is a red flag —
real, independent signals rarely correlate this strongly with an outcome. It's high
because gsc_clicks was literally used to construct the label via a threshold, not
because it's a genuinely powerful predictor.
This is artificially inflated because 'leaky_feature' (gsc_clicks) is the exact
value the label was thresholded from — a real model would 'cheat' by learning this
shortcut instead of a genuine pattern.

STEP 2: Remove the leaky feature and keep only honest, independent features:
Honest feature set: ['gsc_avg_position', 'ga4_engaged_sessions', 'sessions_ai', 'scroll_events']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [15]:
print("This data cannot tell you:")
print()
print("1. GA4 engagement data is missing for ~96% of rows (only 4.2% have ga4_data_available=True).")
print("   This is NOT random — it follows each client's ga4_data_start date. Rows before that")
print("   date are zero-filled, not truly zero-engagement. Any GA4 feature must filter on")
print("   ga4_data_available=True, or it will look like most content gets zero traffic when")
print("   really the data just isn't being collected yet for that client.")
print()
print("2. Only 36.7% of rows have GSC (Search Console) data available — most clients/content")
print("   in this slice simply don't have search visibility data for March 2026.")
print()
print("3. History depth varies wildly per client (55 clients in this month alone) — a global")
print("   March window doesn't mean equal history for every client; some may have started")
print("   reporting mid-month or have gaps.")
print()
print("4. This is one month only (March 2026). It cannot show seasonal patterns, trends over")
print("   time, or whether March performance is typical or unusual for these clients.")
print()
print("5. This month sits BEFORE the query table's 90-day window overlaps it, so joining")
print("   fact_content_query_90d to this month risks pulling in future context unless window")
print("   alignment is checked first.")

This data cannot tell you:

1. GA4 engagement data is missing for ~96% of rows (only 4.2% have ga4_data_available=True).
   This is NOT random — it follows each client's ga4_data_start date. Rows before that
   date are zero-filled, not truly zero-engagement. Any GA4 feature must filter on
   ga4_data_available=True, or it will look like most content gets zero traffic when
   really the data just isn't being collected yet for that client.

2. Only 36.7% of rows have GSC (Search Console) data available — most clients/content
   in this slice simply don't have search visibility data for March 2026.

3. History depth varies wildly per client (55 clients in this month alone) — a global
   March window doesn't mean equal history for every client; some may have started
   reporting mid-month or have gaps.

4. This is one month only (March 2026). It cannot show seasonal patterns, trends over
   time, or whether March performance is typical or unusual for these clients.

5. This month sits BEF

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.